# Creation of the **IberFire** datacube

This notebook will generate the *datacube* for training the forest-fire predictors.

To create the *datacube*, using a ``raster`` file generated with QGIS we first create the base xarray.Dataset and then start adding variables from ``raster`` files. On this notebook, we only add the temporal-only features and the fire-related variables (is_fire, is_near_fire).

This notebook presents the code used to create a part of **IberFire** (without the spatio-temporal features). However, only the first part can be executed locally (up to the is_holiday feature), since all the .tif `raster` files are not available.

---

#### Note:

All the features except is_holiday are added with external .tif `raster` files generated on QGIS. To generate the is_holiday feature, the `holidays` package is used along with the AutonomoousCommunities feature.

## 1. Generate base:

In [1]:
import xarray as xr
import rasterio
import numpy as np
import pandas as pd

from src.config import BASE_RASTER_FILE, AUTONOMOUS_COMMUNITIES_FILE

with rasterio.open(BASE_RASTER_FILE) as raster:
    print("       " + "-"*27)
    print("       " + "----- Raster metadata -----")
    print("       " + "-"*27)
    print()

    # Read the raster data and metadata
    spatial_data = raster.read(1)  # Read the first band
    binary_data = np.where(spatial_data < -1, 0, 1).astype(np.uint16)

    transform = raster.transform
    x_res, y_res = transform.a, -transform.e  # Resolutions 
    xmin, ymax = transform.c, transform.f  # Top-left corner
    print(f"Resolution: {x_res} x {y_res} m")
    print(f"Top-left corner: {xmin}, {ymax}")

    width = raster.width
    height = raster.height
    print(f"Width x Height: {width} x {height}")

    crs = raster.crs # Coordinate Reference System: EPSG:3035
    print(f"CRS: {crs.to_string()}")
    
    spatial_shape = raster.shape  # (rows, cols)
    print(f"Shape: {spatial_shape}")

    # Generate x and y coordinates from the transform
    x_coords = np.arange(xmin, xmin + width * x_res, x_res)
    y_coords = np.arange(ymax, ymax - height * y_res, -y_res)

    print()
    print("-"*43)
    print("-"*43)


# Step 2: Define the temporal range
time_range = pd.date_range(start="2007-12-01", end="2024-12-31", freq="D")

# Step 3: Create the xarray dataset
x_coordinate = np.tile(x_coords, (len(y_coords), 1)).astype(np.float32)  # Repeat x_coords along the rows
y_coordinate = np.tile(y_coords[:, np.newaxis], (1, len(x_coords))).astype(np.float32)  # Repeat y_coords along the columns

x_index = np.tile(np.arange(len(x_coords)), (len(y_coords), 1)).astype(np.uint16)
y_index = np.tile(np.arange(len(y_coords))[:, np.newaxis], (1, len(x_coords))).astype(np.uint16)

ds = xr.Dataset(
    {
        "x_index": (("y", "x"), x_index),
        "y_index": (("y", "x"), y_index),
        "x_coordinate": (("y", "x"), x_coordinate),
        "y_coordinate": (("y", "x"), y_coordinate),
        "is_spain": (("y", "x"), binary_data)
    },
    coords={
        "x": x_coords,
        "y": y_coords,
        "time": time_range,
    },
    attrs={
        "description": "Datacube for Spain with 1km x 1km spatial resolution and daily temporal resolution.",
        "crs": crs.to_string(),
        "spatial_resolution": f"{x_res} x {y_res} (meters)",
        "top_left_corner": f"{xmin}, {ymax}",
    }
)

# Update index
ds = ds.set_index({"time": "time", "x": "x", "y": "y"})

# Add metadata to the variable "is_spain"
ds["is_spain"].attrs = {
    "description": "Binary mask indicating the Spanish region: 1 for Spanish region, 0 otherwise.",
    "valid_values": "{0, 1}",
    "data_type": "uint8",
}

ds["x_coordinate"].attrs = {
    "description": "X-coordinate values in the EPSG:3035 reference system.",
    "units": "meters"
}

ds["x_index"].attrs = {
    "description": "X-coordinate index values.",
    "units": "index"
}


ds["y_coordinate"].attrs = {
    "description": "Y-coordinate values in the EPSG:3035 reference system.",
    "units": "meters"
}

ds["y_index"].attrs = {
    "description": "y-coordinate index values.",
    "units": "index"
}


ds

2025-07-04 09:46:42.087 | INFO     | src.config:<module>:13 - PROJ_ROOT path is: C:\Workspace\Projects\GAIA\Repos\IberFire


       ---------------------------
       ----- Raster metadata -----
       ---------------------------

Resolution: 1000.0 x 1000.0000000000002 m
Top-left corner: 2674734.3466, 2492195.9911
Width x Height: 1188 x 920
CRS: EPSG:3035
Shape: (920, 1188)

-------------------------------------------
-------------------------------------------


<xarray.Dataset> Size: 15MB
Dimensions:       (y: 920, x: 1188, time: 6241)
Coordinates:
  * x             (x) float64 10kB 2.675e+06 2.676e+06 ... 3.861e+06 3.862e+06
  * y             (y) float64 7kB 2.492e+06 2.491e+06 ... 1.574e+06 1.573e+06
  * time          (time) datetime64[ns] 50kB 2007-12-01 ... 2024-12-31
Data variables:
    x_index       (y, x) uint16 2MB 0 1 2 3 4 5 ... 1183 1184 1185 1186 1187
    y_index       (y, x) uint16 2MB 0 0 0 0 0 0 0 ... 919 919 919 919 919 919
    x_coordinate  (y, x) float32 4MB 2.675e+06 2.676e+06 ... 3.861e+06 3.862e+06
    y_coordinate  (y, x) float32 4MB 2.492e+06 2.492e+06 ... 1.573e+06 1.573e+06
    is_spain      (y, x) uint16 2MB 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0
Attributes:
    description:         Datacube for Spain with 1km x 1km spatial resolution...
    crs:                 EPSG:3035
    spatial_resolution:  1000.0 x 1000.0000000000002 (meters)
    top_left_corner:     2674734.3466, 2492195.9911

## 2. Add spatial-only features

### 2.0 Auxiliary / regional features

#### Add AutonomousCommunities feature

In [2]:
with rasterio.open(AUTONOMOUS_COMMUNITIES_FILE) as raster:
    data = raster.read(1).astype(np.uint16)  # Read the first band

    assert raster.width == width and raster.height == height

    ds["AutonomousCommunities"] = (("y", "x"), data)

    ds["AutonomousCommunities"].attrs = {
        "description": f"The Autonomous Communities code. ",
        "source": "Data obtained from https://data.metabolismofcities.org/library/35475/",
        "valid_values": "{1,2, ..., 19}",
        "00": "Nodata",
        "01": "Andalucía",
        "02": "Aragón",
        "03": "Principado de Asturias",
        "04": "Islas Baleares",
        "05": "Canarias",
        "06": "Cantabria",
        "07": "Castilla y León",
        "08": "Castilla - La Mancha",
        "09": "Cataluña",
        "10": "Comunidad Valenciana",
        "11": "Extremadura",
        "12": "Galicia",
        "13": "Comunidad de Madrid",
        "14": "Región de Murcia",
        "15": "Comunidad Foral de Navarra",
        "16": "País Vasco",
        "17": "La Rioja",
        "18": "Ceuta",
        "19": "Melilla"

    }

ds

<xarray.Dataset> Size: 18MB
Dimensions:                (y: 920, x: 1188, time: 6241)
Coordinates:
  * x                      (x) float64 10kB 2.675e+06 2.676e+06 ... 3.862e+06
  * y                      (y) float64 7kB 2.492e+06 2.491e+06 ... 1.573e+06
  * time                   (time) datetime64[ns] 50kB 2007-12-01 ... 2024-12-31
Data variables:
    x_index                (y, x) uint16 2MB 0 1 2 3 4 ... 1184 1185 1186 1187
    y_index                (y, x) uint16 2MB 0 0 0 0 0 0 ... 919 919 919 919 919
    x_coordinate           (y, x) float32 4MB 2.675e+06 2.676e+06 ... 3.862e+06
    y_coordinate           (y, x) float32 4MB 2.492e+06 2.492e+06 ... 1.573e+06
    is_spain               (y, x) uint16 2MB 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0
    AutonomousCommunities  (y, x) uint16 2MB 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0
Attributes:
    description:         Datacube for Spain with 1km x 1km spatial resolution...
    crs:                 EPSG:3035
    spatial_resolution:  1000.0 x 1000.0000000000002 (meters)
    top_left_corner:     2674734.3466, 2492195.9911

#### Add is_holiday feature

In [3]:
import holidays
import pandas as pd

is_holiday_data = np.zeros((len(ds.time), len(ds.y), len(ds.x)), dtype = np.uint8)

ds["is_holiday"] = xr.DataArray(
    data=is_holiday_data,
    coords={
        "time": ds.time,
        "y": ds.y,
        "x": ds.x
    },
    dims=("time", "y", "x"),
    name="is_holiday"
)

ds["is_holiday"].attrs = {
    "description": "Binary mask indicating whether it is holiday in that pixel in that time or not..",
    "valid_values": "{0, 1}",
    "data_type": "uint8",
}

hist_aut_com = {1: "AN",
                2: "AR",
                3: "AS",
                4: "IB",
                5: "CN",
                6: "CB",
                7: "CL",
                8: "CM",
                9: "CT",
                10: "VC",
                11: "EX",
                12: "GA",
                13: "MD",
                14: "MC",
                15: "NC",
                16: "PV",
                17: "RI",
                18: "CE",
                19: "ML"

}

# # Add saturdays and sundays
for i, time in enumerate(ds.time):
    fecha = pd.Timestamp(time.values).date()
    if fecha.weekday() >= 5:
        ds["is_holiday"].values[i,:,:] = 1
        continue

# Add special holidays 2008-2024
for aut_code in range(1,20):
    festivos_com_autonoma = holidays.Spain(years = [year for year in range(2008, 2025)], subdiv=hist_aut_com[aut_code])
    for hol_date in festivos_com_autonoma.keys():
        hol_date = pd.Timestamp(hol_date)
        time_index = t_idx = int(np.where(ds.time.values == hol_date)[0][0])
        region_mask = ds["AutonomousCommunities"].data == aut_code
        ds["is_holiday"].data[t_idx, region_mask] = 1


In [4]:
ds

<xarray.Dataset> Size: 7GB
Dimensions:                (y: 920, x: 1188, time: 6241)
Coordinates:
  * x                      (x) float64 10kB 2.675e+06 2.676e+06 ... 3.862e+06
  * y                      (y) float64 7kB 2.492e+06 2.491e+06 ... 1.573e+06
  * time                   (time) datetime64[ns] 50kB 2007-12-01 ... 2024-12-31
Data variables:
    x_index                (y, x) uint16 2MB 0 1 2 3 4 ... 1184 1185 1186 1187
    y_index                (y, x) uint16 2MB 0 0 0 0 0 0 ... 919 919 919 919 919
    x_coordinate           (y, x) float32 4MB 2.675e+06 2.676e+06 ... 3.862e+06
    y_coordinate           (y, x) float32 4MB 2.492e+06 2.492e+06 ... 1.573e+06
    is_spain               (y, x) uint16 2MB 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0
    AutonomousCommunities  (y, x) uint16 2MB 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0
    is_holiday             (time, y, x) uint8 7GB 1 1 1 1 1 1 1 ... 0 0 0 0 0 0
Attributes:
    description:         Datacube for Spain with 1km x 1km spatial resolution...
    crs:                 EPSG:3035
    spatial_resolution:  1000.0 x 1000.0000000000002 (meters)
    top_left_corner:     2674734.3466, 2492195.9911

---
---
---

### Note:
Below here, the code can not be executed since all the necessary files are not available.

---
---
---

#### Is_sea, Is_waterbody

In [ ]:
import os

path = "....../Datacube_variables/"
file = path + "Is_europe.tif"

with rasterio.open(file) as raster:
    data = raster.read(1)  # Read the first band

    assert raster.width == width and raster.height == height

    ds["is_europe"] = (("y", "x"), data) # A lower resolution raster than Is_land, but with no "NA" values inside

ds

<xarray.Dataset> Size: 7GB
Dimensions:                (y: 920, x: 1188, time: 6241)
Coordinates:
  * x                      (x) float64 10kB 2.675e+06 2.676e+06 ... 3.862e+06
  * y                      (y) float64 7kB 2.492e+06 2.491e+06 ... 1.573e+06
  * time                   (time) datetime64[ns] 50kB 2007-12-01 ... 2024-12-31
Data variables:
    x_index                (y, x) uint16 2MB 0 1 2 3 4 ... 1184 1185 1186 1187
    y_index                (y, x) uint16 2MB 0 0 0 0 0 0 ... 919 919 919 919 919
    x_coordinate           (y, x) float32 4MB 2.675e+06 2.676e+06 ... 3.862e+06
    y_coordinate           (y, x) float32 4MB 2.492e+06 2.492e+06 ... 1.573e+06
    is_spain               (y, x) uint16 2MB 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0
    AutonomousCommunities  (y, x) uint16 2MB 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0
    is_holiday             (time, y, x) uint8 7GB 1 1 1 1 1 1 1 ... 0 0 0 0 0 0
    is_europe              (y, x) float32 4MB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0
Attributes:
    description:         Datacube for Spain with 1km x 1km spatial resolution...
    crs:                 EPSG:3035
    spatial_resolution:  1000.0 x 1000.0000000000002 (meters)
    top_left_corner:     2674734.3466, 2492195.9911

In [ ]:
import os

file = path + "IsLand_province.tif"

with rasterio.open(file) as raster:
    data = raster.read(1)  # Read the first band

    assert raster.width == width and raster.height == height

    ds["is_sea"] = (("y", "x"), data)
    
    ds["is_sea"].attrs = {
        "description": f"Whether it is sea or not. The data was obtained by processing the Copernicus Global Digital Elevation Model",
        "source": "European Space Agency (2024).  Copernicus Global Digital Elevation Model.  Distributed by OpenTopography.  https://doi.org/10.5069/G9028PQB. Accessed: 2025-01-14",
        "valid_values": "{0,1}"
    }

    ds["is_sea"].values = np.where(ds["is_sea"] == 1, 0, 1).astype(np.uint8) # Make it "is_sea" indeed

    ds["is_sea"].values = np.where((ds["is_europe"].values == 1) & (ds["is_sea"].values == 1), 0, ds["is_sea"]).astype(np.uint8) # Make in land water to 0

    ds["is_waterbody"] = (("y", "x"), 1 - data)
    
    ds["is_waterbody"].attrs = {
        "description": f"Whether it is a big water body or not. The data was obtained by processing the Copernicus Global Digital Elevation Model",
        "source": "European Space Agency (2024).  Copernicus Global Digital Elevation Model.  Distributed by OpenTopography.  https://doi.org/10.5069/G9028PQB. Accessed: 2025-01-14",
        "valid_values": "{0,1}"
    }

ds = ds.drop_vars("is_europe")
ds

<xarray.Dataset> Size: 7GB
Dimensions:                (y: 920, x: 1188, time: 6241)
Coordinates:
  * x                      (x) float64 10kB 2.675e+06 2.676e+06 ... 3.862e+06
  * y                      (y) float64 7kB 2.492e+06 2.491e+06 ... 1.573e+06
  * time                   (time) datetime64[ns] 50kB 2007-12-01 ... 2024-12-31
Data variables:
    x_index                (y, x) uint16 2MB 0 1 2 3 4 ... 1184 1185 1186 1187
    y_index                (y, x) uint16 2MB 0 0 0 0 0 0 ... 919 919 919 919 919
    x_coordinate           (y, x) float32 4MB 2.675e+06 2.676e+06 ... 3.862e+06
    y_coordinate           (y, x) float32 4MB 2.492e+06 2.492e+06 ... 1.573e+06
    is_spain               (y, x) uint16 2MB 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0
    AutonomousCommunities  (y, x) uint16 2MB 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0
    is_holiday             (time, y, x) uint8 7GB 1 1 1 1 1 1 1 ... 0 0 0 0 0 0
    is_sea                 (y, x) uint8 1MB 1 1 1 1 1 1 1 1 ... 1 1 1 1 1 1 1 1
    is_waterbody           (y, x) int32 4MB 1 1 1 1 1 1 1 1 ... 1 1 1 1 1 1 1 1
Attributes:
    description:         Datacube for Spain with 1km x 1km spatial resolution...
    crs:                 EPSG:3035
    spatial_resolution:  1000.0 x 1000.0000000000002 (meters)
    top_left_corner:     2674734.3466, 2492195.9911

### 2.1 Corine Land Cover

The processed CLC variables with the same shape as the datacube created above.

In [6]:
# 5 classes
label1_dict = {
    1: "Artificial surfaces",
    2: "Artificial surfaces",
    3: "Artificial surfaces",
    4: "Artificial surfaces",
    5: "Artificial surfaces",
    6: "Artificial surfaces",
    7: "Artificial surfaces",
    8: "Artificial surfaces",
    9: "Artificial surfaces",
    10: "Artificial surfaces",
    11: "Artificial surfaces",
    12: "Agricultural areas",
    13: "Agricultural areas",
    14: "Agricultural areas",
    15: "Agricultural areas",
    16: "Agricultural areas",
    17: "Agricultural areas",
    18: "Agricultural areas",
    19: "Agricultural areas",
    20: "Agricultural areas",
    21: "Agricultural areas",
    22: "Agricultural areas",
    23: "Forest and semi natural areas",
    24: "Forest and semi natural areas",
    25: "Forest and semi natural areas",
    26: "Forest and semi natural areas",
    27: "Forest and semi natural areas",
    28: "Forest and semi natural areas",
    29: "Forest and semi natural areas",
    30: "Forest and semi natural areas",
    31: "Forest and semi natural areas",
    32: "Forest and semi natural areas",
    33: "Forest and semi natural areas",
    34: "Forest and semi natural areas",
    35: "Wetlands",
    36: "Wetlands",
    37: "Wetlands",
    38: "Wetlands",
    39: "Wetlands",
    40: "Water bodies",
    41: "Water bodies",
    42: "Water bodies",
    43: "Water bodies",
    44: "Water bodies",
    48: "NODATA",
    "NODATA": "NODATA"
}

# 15 classes
label2_dict = {
    1: "Urban fabric",
    2: "Urban fabric",
    3: "Industrial, commercial and transport units",
    4: "Industrial, commercial and transport units",
    5: "Industrial, commercial and transport units",
    6: "Industrial, commercial and transport units",
    7: "Mine, dump and construction sites",
    8: "Mine, dump and construction sites",
    9: "Mine, dump and construction sites",
    10: "Artificial, non-agricultural vegetated areas",
    11: "Artificial, non-agricultural vegetated areas",
    12: "Arable land",
    13: "Arable land",
    14: "Arable land",
    15: "Permanent crops",
    16: "Permanent crops",
    17: "Permanent crops",
    18: "Pastures",
    19: "Heterogeneous agricultural areas",
    20: "Heterogeneous agricultural areas",
    21: "Heterogeneous agricultural areas",
    22: "Heterogeneous agricultural areas",
    23: "Forests",
    24: "Forests",
    25: "Forests",
    26: "Scrub and/or herbaceous vegetation associations",
    27: "Scrub and/or herbaceous vegetation associations",
    28: "Scrub and/or herbaceous vegetation associations",
    29: "Scrub and/or herbaceous vegetation associations",
    30: "Open spaces with little or no vegetation",
    31: "Open spaces with little or no vegetation",
    32: "Open spaces with little or no vegetation",
    33: "Open spaces with little or no vegetation",
    34: "Open spaces with little or no vegetation",
    35: "Inland wetlands",
    36: "Inland wetlands",
    37: "Maritime wetlands",
    38: "Maritime wetlands",
    39: "Maritime wetlands",
    40: "Inland waters",
    41: "Inland waters",
    42: "Marine waters",
    43: "Marine waters",
    44: "Marine waters",
    48: "NODATA",
    "NODATA": "NODATA"
}

# 44 classes
label3_dict = {
    1: "Continuous urban fabric",
    2: "Discontinuous urban fabric",
    3: "Industrial or commercial units",
    4: "Road and rail networks and associated land",
    5: "Port areas",
    6: "Airports",
    7: "Mineral extraction sites",
    8: "Dump sites",
    9: "Construction sites",
    10: "Green urban areas",
    11: "Sport and leisure facilities",
    12: "Non-irrigated arable land",
    13: "Permanently irrigated land",
    14: "Rice fields",
    15: "Vineyards",
    16: "Fruit trees and berry plantations",
    17: "Olive groves",
    18: "Pastures",
    19: "Annual crops associated with permanent crops",
    20: "Complex cultivation patterns",
    21: "Land principally occupied by agriculture, with significant areas of natural vegetation",
    22: "Agro-forestry areas",
    23: "Broad-leaved forest",
    24: "Coniferous forest",
    25: "Mixed forest",
    26: "Natural grasslands",
    27: "Moors and heathland",
    28: "Sclerophyllous vegetation",
    29: "Transitional woodland-shrub",
    30: "Beaches, dunes, sands",
    31: "Bare rocks",
    32: "Sparsely vegetated areas",
    33: "Burnt areas",
    34: "Glaciers and perpetual snow",
    35: "Inland marshes",
    36: "Peat bogs",
    37: "Salt marshes",
    38: "Salines",
    39: "Intertidal flats",
    40: "Water courses",
    41: "Water bodies",
    42: "Coastal lagoons",
    43: "Estuaries",
    44: "Sea and ocean",
    48: "NODATA",
    "NODATA": "NODATA"
}



In [ ]:
import os

tiff_files = [f"CLC_{year}_{cl}.tif" for year in [2006, 2012, 2018] for cl in range(1,45)]
tiff_files += [f"CLC_{year}_NODATA.tif" for year in [2006, 2012, 2018]]

for tiff_file in tiff_files:
    variable_name = os.path.splitext(tiff_file)[0]  # Use file name (without extension) as variable name
    file_path = os.path.join(path, tiff_file)
    class_number = variable_name.split("_")[-1]
    year = variable_name.split("_")[1]

    with rasterio.open(file_path) as raster:
        data = raster.read(1)  # Read the first band

        assert raster.width == width and raster.height == height

        ds[variable_name] = (("y", "x"), data)
        
        ds[variable_name].attrs = {
            "description": f"Proportion of the class {class_number} in {year} in the 1km x 1km grid cell",
            "source": "https://land.copernicus.eu/en/products/corine-land-cover",
            "valid_range": "[0,1]",
            "units": "proportion",
            "label class of CLC": 3,
            "label_1":label1_dict[int(class_number) if class_number != "NODATA" else class_number],
            "label_2":label2_dict[int(class_number) if class_number != "NODATA" else class_number],
            "label_3":label3_dict[int(class_number) if class_number != "NODATA" else class_number],
        }

ds

<xarray.Dataset> Size: 7GB
Dimensions:                (y: 920, x: 1188, time: 6241)
Coordinates:
  * x                      (x) float64 10kB 2.675e+06 2.676e+06 ... 3.862e+06
  * y                      (y) float64 7kB 2.492e+06 2.491e+06 ... 1.573e+06
  * time                   (time) datetime64[ns] 50kB 2007-12-01 ... 2024-12-31
Data variables: (12/144)
    x_index                (y, x) uint16 2MB 0 1 2 3 4 ... 1184 1185 1186 1187
    y_index                (y, x) uint16 2MB 0 0 0 0 0 0 ... 919 919 919 919 919
    x_coordinate           (y, x) float32 4MB 2.675e+06 2.676e+06 ... 3.862e+06
    y_coordinate           (y, x) float32 4MB 2.492e+06 2.492e+06 ... 1.573e+06
    is_spain               (y, x) uint16 2MB 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0
    AutonomousCommunities  (y, x) uint16 2MB 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0
    ...                     ...
    CLC_2018_42            (y, x) float32 4MB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0
    CLC_2018_43            (y, x) float32 4MB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0
    CLC_2018_44            (y, x) float32 4MB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0
    CLC_2006_NODATA        (y, x) float32 4MB 100.0 100.0 100.0 ... 100.0 100.0
    CLC_2012_NODATA        (y, x) float32 4MB 100.0 100.0 100.0 ... 100.0 100.0
    CLC_2018_NODATA        (y, x) float32 4MB 100.0 100.0 100.0 ... 100.0 100.0
Attributes:
    description:         Datacube for Spain with 1km x 1km spatial resolution...
    crs:                 EPSG:3035
    spatial_resolution:  1000.0 x 1000.0000000000002 (meters)
    top_left_corner:     2674734.3466, 2492195.9911

Transform the CLC to generate the proportions and clip to the Spanish area. Drop unnecesary variables.

In [8]:
import matplotlib.pyplot as plt

years = [2006, 2012, 2018]

for year in years:
    for cl in range(1,45):
        ds[f"CLC_{year}_{cl}"].values = ds[f"CLC_{year}_{cl}"].values / 100 # Convert to proportion
        ds[f"CLC_{year}_{cl}"].values = ds[f"CLC_{year}_{cl}"].where(ds["is_sea"] == 0, 0)  # Mask out non-Spain areas as proportion = 0
    ds[f"CLC_{year}_44"].values = ds[f"CLC_{year}_44"].where(ds["is_sea"] == 0, 1) # Mask out sea areas as proportion = 1 for class 44 (sea and ocean class)

variables_to_drop = [f"CLC_{year}_{cl}" for year in years for cl in ["NODATA"]]

# Drop the variables from the dataset
ds = ds.drop_vars(variables_to_drop)

ds


<xarray.Dataset> Size: 7GB
Dimensions:                (y: 920, x: 1188, time: 6241)
Coordinates:
  * x                      (x) float64 10kB 2.675e+06 2.676e+06 ... 3.862e+06
  * y                      (y) float64 7kB 2.492e+06 2.491e+06 ... 1.573e+06
  * time                   (time) datetime64[ns] 50kB 2007-12-01 ... 2024-12-31
Data variables: (12/141)
    x_index                (y, x) uint16 2MB 0 1 2 3 4 ... 1184 1185 1186 1187
    y_index                (y, x) uint16 2MB 0 0 0 0 0 0 ... 919 919 919 919 919
    x_coordinate           (y, x) float32 4MB 2.675e+06 2.676e+06 ... 3.862e+06
    y_coordinate           (y, x) float32 4MB 2.492e+06 2.492e+06 ... 1.573e+06
    is_spain               (y, x) uint16 2MB 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0
    AutonomousCommunities  (y, x) uint16 2MB 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0
    ...                     ...
    CLC_2018_39            (y, x) float32 4MB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0
    CLC_2018_40            (y, x) float32 4MB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0
    CLC_2018_41            (y, x) float32 4MB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0
    CLC_2018_42            (y, x) float32 4MB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0
    CLC_2018_43            (y, x) float32 4MB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0
    CLC_2018_44            (y, x) float32 4MB 1.0 1.0 1.0 1.0 ... 1.0 1.0 1.0
Attributes:
    description:         Datacube for Spain with 1km x 1km spatial resolution...
    crs:                 EPSG:3035
    spatial_resolution:  1000.0 x 1000.0000000000002 (meters)
    top_left_corner:     2674734.3466, 2492195.9911

Add some new variables to the dataset (combinations of CLC variables):

In [9]:
for year in [2006, 2012, 2018]:
    urban_fabric_proportion = sum(ds[var] for var in [f"CLC_{year}_{i}" for i in range(1,3)])
    industrial_proportion = sum(ds[var] for var in [f"CLC_{year}_{i}" for i in range(3,7)])
    mine_proportion = sum(ds[var] for var in [f"CLC_{year}_{i}" for i in range(7,10)])
    artificial_vegetation_proportion = sum(ds[var] for var in [f"CLC_{year}_{i}" for i in range(10,12)])
    arable_land_proportion = sum(ds[var] for var in [f"CLC_{year}_{i}" for i in range(12,15)])
    permanent_crops_proportion = sum(ds[var] for var in [f"CLC_{year}_{i}" for i in range(15,18)])
    heterogeneous_agriculture_proportion = sum(ds[var] for var in [f"CLC_{year}_{i}" for i in range(19,23)])
    forest_proportion = sum(ds[var] for var in [f"CLC_{year}_{i}" for i in range(23,26)])
    scrub_proportion = sum(ds[var] for var in [f"CLC_{year}_{i}" for i in range(26,30)])
    open_space_proportion = sum(ds[var] for var in [f"CLC_{year}_{i}" for i in range(30,35)])
    inland_wetlands_proportion = sum(ds[var] for var in [f"CLC_{year}_{i}" for i in range(35,37)])
    maritime_wetlands_proportion = sum(ds[var] for var in [f"CLC_{year}_{i}" for i in range(37,40)])
    inland_waters_proportion = sum(ds[var] for var in [f"CLC_{year}_{i}" for i in range(40,42)])
    marine_waters_proportion = sum(ds[var] for var in [f"CLC_{year}_{i}" for i in range(42,45)])

    ds[f"CLC_{year}_urban_fabric_proportion"] = urban_fabric_proportion
    ds[f"CLC_{year}_industrial_proportion"] = industrial_proportion
    ds[f"CLC_{year}_mine_proportion"] = mine_proportion
    ds[f"CLC_{year}_artificial_vegetation_proportion"] = artificial_vegetation_proportion
    ds[f"CLC_{year}_arable_land_proportion"] = arable_land_proportion
    ds[f"CLC_{year}_permanent_crops_proportion"] = permanent_crops_proportion
    ds[f"CLC_{year}_heterogeneous_agriculture_proportion"] = heterogeneous_agriculture_proportion
    ds[f"CLC_{year}_forest_proportion"] = forest_proportion
    ds[f"CLC_{year}_scrub_proportion"] = scrub_proportion
    ds[f"CLC_{year}_open_space_proportion"] = open_space_proportion
    ds[f"CLC_{year}_inland_wetlands_proportion"] = inland_wetlands_proportion
    ds[f"CLC_{year}_maritime_wetlands_proportion"] = maritime_wetlands_proportion
    ds[f"CLC_{year}_inland_waters_proportion"] = inland_waters_proportion
    ds[f"CLC_{year}_marine_waters_proportion"] = marine_waters_proportion
    
    # Add metadata
    ds[f"CLC_{year}_urban_fabric_proportion"].attrs = {
        "description": f"Proportion of urban fabric for the year {year}. Summed classes: {', '.join([str(i) for i in range(1,3)])}",
        "units": "proportion",
        "label class of CLC": 2
    }
    ds[f"CLC_{year}_industrial_proportion"].attrs = {
        "description": f"Proportion of industrial, commercial and transport units areas for the year {year}. Summed classes: {', '.join([str(i) for i in range(3,7)])}",
        "units": "proportion",
        "label class of CLC": 2
    }
    ds[f"CLC_{year}_mine_proportion"].attrs = {
        "description": f"Proportion of mine, dump and construction sites for the year {year}. Summed classes: {', '.join([str(i) for i in range(7,10)])}",
        "units": "proportion",
        "label class of CLC": 2
    }
    ds[f"CLC_{year}_artificial_vegetation_proportion"].attrs = {
        "description": f"Proportion of artificial, non-agricultural vegetated areas for the year {year}. Summed classes: {', '.join([str(i) for i in range(10,12)])}",
        "units": "proportion",
        "label class of CLC": 2
    }
    ds[f"CLC_{year}_arable_land_proportion"].attrs = {
        "description": f"Proportion of arable land for the year {year}. Summed classes: {', '.join([str(i) for i in range(12,15)])}",
        "units": "proportion",
        "label class of CLC": 2
    }
    ds[f"CLC_{year}_permanent_crops_proportion"].attrs = {
        "description": f"Proportion of permanent crops for the year {year}. Summed classes: {', '.join([str(i) for i in range(15,18)])}",
        "units": "proportion",
        "label class of CLC": 2
    }
    ds[f"CLC_{year}_heterogeneous_agriculture_proportion"].attrs = {
        "description": f"Proportion of heterogeneous agricultural areas for the year {year}. Summed classes: {', '.join([str(i) for i in range(19,23)])}",
        "units": "proportion",
        "label class of CLC": 2
    }
    ds[f"CLC_{year}_forest_proportion"].attrs = {
        "description": f"Proportion of forests for the year {year}. Summed classes: {', '.join([str(i) for i in range(23,26)])}",
        "units": "proportion",
        "label class of CLC": 2
    }
    ds[f"CLC_{year}_scrub_proportion"].attrs = {
        "description": f"Proportion of scrub and/or herbaceous vegetation associations for the year {year}. Summed classes: {', '.join([str(i) for i in range(26,30)])}",
        "units": "proportion",
        "label class of CLC": 2
    }
    ds[f"CLC_{year}_open_space_proportion"].attrs = {
        "description": f"Proportion of open spaces with little or no vegetation for the year {year}. Summed classes: {', '.join([str(i) for i in range(30,35)])}",
        "units": "proportion",
        "label class of CLC": 2
    }
    ds[f"CLC_{year}_inland_wetlands_proportion"].attrs = {
        "description": f"Proportion of inland wetlands for the year {year}. Summed classes: {', '.join([str(i) for i in range(35,37)])}",
        "units": "proportion",
        "label class of CLC": 2
    }
    ds[f"CLC_{year}_maritime_wetlands_proportion"].attrs = {
        "description": f"Proportion of maritime wetlands for the year {year}. Summed classes: {', '.join([str(i) for i in range(37,40)])}",
        "units": "proportion",
        "label class of CLC": 2
    }
    ds[f"CLC_{year}_inland_waters_proportion"].attrs = {
        "description": f"Proportion of inland waters for the year {year}. Summed classes: {', '.join([str(i) for i in range(40,42)])}",
        "units": "proportion",
        "label class of CLC": 2
    }
    ds[f"CLC_{year}_marine_waters_proportion"].attrs = {
        "description": f"Proportion of marine waters for the year {year}. Summed classes: {', '.join([str(i) for i in range(42,45)])}",
        "units": "proportion",
        "label class of CLC": 2
    }

In [10]:
for year in [2006, 2012, 2018]:
    artificial_proportion = sum(ds[var] for var in [f"CLC_{year}_{i}" for i in range(1,12)])
    agricultural_proportion = sum(ds[var] for var in [f"CLC_{year}_{i}" for i in range(12,23)])
    forest_semi_natural_proportion = sum(ds[var] for var in [f"CLC_{year}_{i}" for i in range(23,35)])
    wetlands_proportion = sum(ds[var] for var in [f"CLC_{year}_{i}" for i in range(35,40)])
    waterbody_proportion = sum(ds[var] for var in [f"CLC_{year}_{i}" for i in range(40,45)])
    
    ds[f"CLC_{year}_artificial_proportion"] = artificial_proportion
    ds[f"CLC_{year}_agricultural_proportion"] = agricultural_proportion
    ds[f"CLC_{year}_forest_and_semi_natural_proportion"] = forest_semi_natural_proportion
    ds[f"CLC_{year}_wetlands_proportion"] = wetlands_proportion
    ds[f"CLC_{year}_waterbody_proportion"] = waterbody_proportion

    # Add metadata
    ds[f"CLC_{year}_artificial_proportion"].attrs = {
        "description": f"Proportion of artificial surfaces for the year {year}. Summed classes: {', '.join([str(i) for i in range(1,12)])}",
        "units": "proportion",
        "label class of CLC": 1
    }
    ds[f"CLC_{year}_agricultural_proportion"].attrs = {
        "description": f"Proportion of agricultural areas for the year {year}. Summed classes: {', '.join([str(i) for i in range(12,23)])}",
        "units": "proportion",
        "label class of CLC": 1
    }
    ds[f"CLC_{year}_forest_and_semi_natural_proportion"].attrs = {
        "description": f"Proportion of forest and semi-natural areas for the year {year}. Summed classes: {', '.join([str(i) for i in range(23,35)])}",
        "units": "proportion",
        "label class of CLC": 1
    }
    ds[f"CLC_{year}_wetlands_proportion"].attrs = {
        "description": f"Proportion of wetlands for the year {year}. Summed classes: {', '.join([str(i) for i in range(35,40)])}",
        "units": "proportion",
        "label class of CLC": 1
    }
    ds[f"CLC_{year}_waterbody_proportion"].attrs = {
        "description": f"Proportion of water bodies for the year {year}. Summed classes: {', '.join([str(i) for i in range(40,45)])}",
        "units": "proportion",
        "label class of CLC": 1
    }

### 2.2 Copernicus DEM

#### Aspect

In [ ]:
import os

aspect_files = [f"aspect_{i}.tif" for i in range(1,9)] + ["aspect_NODATA.tif"]

descriptions_dict = {"NODATA": "No data of aspect available",
                     1: "0-45 degrees",
                     2: "45-90 degrees",
                     3: "90-135 degrees",
                     4: "135-180 degrees",
                     5: "180-225 degrees",
                     6: "225-270 degrees",
                     7: "270-315 degrees",
                     8: "315-360 degrees"}

for tiff_file in aspect_files:
    variable_name = os.path.splitext(tiff_file)[0]  # Use file name (without extension) as variable name
    file_path = os.path.join(path, tiff_file)
    class_number = variable_name.split("_")[-1]

    with rasterio.open(file_path) as raster:
        data = raster.read(1)  # Read the first band

        assert raster.width == width and raster.height == height

        ds[variable_name] = (("y", "x"), data)
        
        ds[variable_name].attrs = {
            "description": f"Proportion of the aspect class {class_number} in the 1km x 1km grid cell. The aspect classes are computed from the digital elevation model (DEM) and represent the orientation of the terrain.",
            f"Class_{class_number}_description": descriptions_dict[int(class_number) if class_number != "NODATA" else class_number],
            "source": "European Space Agency (2024).  Copernicus Global Digital Elevation Model.  Distributed by OpenTopography.  https://doi.org/10.5069/G9028PQB. Accessed: 2025-01-14",
            "valid_range": "[0,1]",
            "units": "proportion",
        }

# Transform the variable into proportions:
variables_aspect = [f"aspect_{i}" for i in range(1,9)] + ["aspect_NODATA"] 
sum_aspects = sum([ds[var] for var in variables_aspect])
for var_name in variables_aspect:
    ds[var_name].values = (ds[var_name] / sum_aspects).values
    ds[var_name].values = ds[var_name].where(ds["is_sea"] == 0, 0)
    ds[var_name] = ds[var_name].fillna(0)


# Add NODATA values = 1 in the "no-Spain" areas (sea areas)
ds["aspect_NODATA"].values = ds["aspect_NODATA"].where(ds["is_sea"] == 0, 1)
ds["aspect_NODATA"] = ds["aspect_NODATA"].fillna(0)

#### Elevation

In [ ]:
import os

elevation_files = [f"elevation_{case}.tif" for case in ["mean", "stdev"]]


for tiff_file in elevation_files:
    variable_name = os.path.splitext(tiff_file)[0]  # Use file name (without extension) as variable name
    file_path = os.path.join(path, tiff_file)
    case = variable_name.split("_")[-1]

    with rasterio.open(file_path) as raster:
        data = raster.read(1)  # Read the first band

        assert raster.width == width and raster.height == height

        ds[variable_name] = (("y", "x"), data)
        
        ds[variable_name].attrs = {
            "description": f"{case} elevation in the 1km x 1km grid cell. The elevation is computed from the digital elevation model (DEM) at 30m resolution.",
            "source": "European Space Agency (2024).  Copernicus Global Digital Elevation Model.  Distributed by OpenTopography.  https://doi.org/10.5069/G9028PQB. Accessed: 2025-01-14",
            "values_approximate_range": "[-10,3500]" if case == "mean" else "[0, 360]",
            "units": "meters",
        }

        ds[variable_name].values = ds[variable_name].where(ds["is_sea"] == 0, 0) # Clip the variable to Spain region


#### Slope

In [ ]:
import os

slope_files = [f"slope_{case}.tif" for case in ["mean", "stdev"]]


for tiff_file in slope_files:
    variable_name = os.path.splitext(tiff_file)[0]  # Use file name (without extension) as variable name
    file_path = os.path.join(path, tiff_file)
    case = variable_name.split("_")[-1]

    with rasterio.open(file_path) as raster:
        data = raster.read(1)  # Read the first band

        assert raster.width == width and raster.height == height

        ds[variable_name] = (("y", "x"), data)
        
        ds[variable_name].attrs = {
            "description": f"{case} slope in the 1km x 1km grid cell. The slope is computed from the digital elevation model (DEM) at 30m resolution.",
            "source": "European Space Agency (2024).  Copernicus Global Digital Elevation Model.  Distributed by OpenTopography.  https://doi.org/10.5069/G9028PQB. Accessed: 2025-01-14",
            "values_approximate_range": "[0,50]" if case == "mean" else "[0, 25]",
            "units": "degrees",
        }

        ds[variable_name].values = ds[variable_name].where(ds["is_sea"] == 0, 0) # Clip the variable to Spain region

#### Roughness

In [ ]:
import os

roughness_files = [f"roughness_{case}.tif" for case in ["mean", "stdev"]]


for tiff_file in roughness_files:
    variable_name = os.path.splitext(tiff_file)[0]  # Use file name (without extension) as variable name
    file_path = os.path.join(path, tiff_file)
    case = variable_name.split("_")[-1]

    with rasterio.open(file_path) as raster:
        data = raster.read(1)  # Read the first band

        assert raster.width == width and raster.height == height

        ds[variable_name] = (("y", "x"), data)
        
        ds[variable_name].attrs = {
            "description": f"{case} roughness in the 1km x 1km grid cell. The slope is computed from the digital elevation model (DEM) at 30m resolution.",
            "source": "European Space Agency (2024).  Copernicus Global Digital Elevation Model.  Distributed by OpenTopography.  https://doi.org/10.5069/G9028PQB. Accessed: 2025-01-14",
            "values_approximate_range": "[0,90]" if case == "mean" else "[0, 80]",
        }

        ds[variable_name].values = ds[variable_name].where(ds["is_sea"] == 0, 0) # Clip the variable to Spain region

### 2.3 Distance to roads

In [ ]:
import os

dist_files = [f"dist_to_{case1}_{case2}.tif" for case1 in ["roads"] for case2 in ["mean", "stdev"]]

for tiff_file in dist_files:
    variable_name = os.path.splitext(tiff_file)[0]  # Use file name (without extension) as variable name
    file_path = os.path.join(path, tiff_file)
    case2 = variable_name.split("_")[-1]
    case1 = variable_name.split("_")[-2]

    with rasterio.open(file_path) as raster:
        data = raster.read(1)  # Read the first band

        assert raster.width == width and raster.height == height

        ds[variable_name] = (("y", "x"), data)
        
        ds[variable_name].attrs = {
            "description": f"{case2} dist to {case1} in the 1km x 1km grid cell.",
            "source": "WorldPop: https://hub.worldpop.org/geodata/summary?id=17504. The layer was obtained with Open Street Map data from 2016",
            "units": "km",
        }

        ds[variable_name].values = ds[variable_name].where(ds["is_spain"] == 1, ds[variable_name].values.mean()) # Clip the variable to Spain region, add average mean distance and stdv in no-spain (and sea) areas

### Distance to waterways

In [ ]:
import os

dist_files = [f"dist_to_{case1}_{case2}.tif" for case1 in ["waterways"] for case2 in ["mean", "stdev"]]

for tiff_file in dist_files:
    variable_name = os.path.splitext(tiff_file)[0]  # Use file name (without extension) as variable name
    file_path = os.path.join(path, tiff_file)
    case2 = variable_name.split("_")[-1]
    case1 = variable_name.split("_")[-2]

    with rasterio.open(file_path) as raster:
        data = raster.read(1)  # Read the first band

        assert raster.width == width and raster.height == height

        ds[variable_name] = (("y", "x"), data)
        
        ds[variable_name].attrs = {
            "description": f"{case2} dist to {case1} in the 1km x 1km grid cell.",
            "source": "WorldPop: https://hub.worldpop.org/geodata/summary?id=18002. The layer was obtained with Open Street Map data from 2016",
            "units": "km",
        }

        ds[variable_name].values = ds[variable_name].where(ds["is_spain"] == 1, ds[variable_name].values.mean()) # Clip the variable to Spain region, add average mean distance and stdv in no-spain (and sea) areas

### Distance to railways

In [ ]:
import os

dist_files = [f"dist_to_{case1}_{case2}.tif" for case1 in ["railways"] for case2 in ["mean", "stdev"]]


for tiff_file in dist_files:
    variable_name = os.path.splitext(tiff_file)[0]  # Use file name (without extension) as variable name
    file_path = os.path.join(path, tiff_file)
    case2 = variable_name.split("_")[-1]
    case1 = variable_name.split("_")[-2]

    with rasterio.open(file_path) as raster:
        data = raster.read(1)  # Read the first band

        assert raster.width == width and raster.height == height

        ds[variable_name] = (("y", "x"), data)
        
        ds[variable_name].attrs = {
            "description": f"{case2} dist to {case1} in the 1km x 1km grid cell.",
            "source": "Open Street Map data from 2024",
            "Note": "The distances are computed from the Open Street Map data. The units of the distance does not represent the physical distance in meters, but a relative distance.",
        }

        ds[variable_name].values = ds[variable_name].where(ds["is_spain"] == 1, ds[variable_name].values.mean()) # Clip the variable to Spain region, add average mean distance and stdv in no-spain (and sea) areas

### 2.4 Natura2000 proportion

In [ ]:
import os

natura2000_files = ["Natura2000_1.tif"]


for tiff_file in natura2000_files:
    variable_name = "is_natura2000"
    file_path = os.path.join(path, tiff_file)

    with rasterio.open(file_path) as raster:
        data = raster.read(1)  # Read the first band

        assert raster.width == width and raster.height == height

        binary_data = np.where(data > 0, 1, 0).astype(np.uint8)

        ds[variable_name] = (("y", "x"), binary_data)
        
        ds[variable_name].attrs = {
            "description": f"Whether the cell is part of the Natura 2000 network. The value 1 indicates that the cell is part of the Natura 2000 network, 0 otherwise.",
            "source": "Spanish government. https://www.miteco.gob.es/es/biodiversidad/servicios/banco-datos-naturaleza/informacion-disponible/rednatura_2000_desc.html"
        }


### 2.5 Population density

In [ ]:
import os

popdens_files = [f"popdens_{year}_{case}.tif" for year in range(2008,2021) for case in ["mean"]]

for tiff_file in popdens_files:
    variable_name = os.path.splitext(tiff_file)[0]  # Use file name (without extension) as variable name
    file_path = os.path.join(path, tiff_file)

    year = variable_name.split("_")[1]
    case = variable_name.split("_")[-1]

    with rasterio.open(file_path) as raster:
        data = raster.read(1)  # Read the first band

        assert raster.width == width and raster.height == height

        ds[variable_name] = (("y", "x"), data)
        
        ds[variable_name].attrs = {
            "description": f"{case} population density in the 1km x 1km grid cell for the year {year}.",
            "source": "WorldPop. https://hub.worldpop.org",
            "units": "people/km^2",
        }

        ds[variable_name].values = ds[variable_name].where(ds["is_sea"] == 0, 0) # Clip the variable to Spain region

In [20]:
ds

<xarray.Dataset> Size: 8GB
Dimensions:                                        (y: 920, x: 1188, time: 6241)
Coordinates:
  * x                                              (x) float64 10kB 2.675e+06...
  * y                                              (y) float64 7kB 2.492e+06 ...
  * time                                           (time) datetime64[ns] 50kB ...
Data variables: (12/233)
    x_index                                        (y, x) uint16 2MB 0 ... 1187
    y_index                                        (y, x) uint16 2MB 0 0 ... 919
    x_coordinate                                   (y, x) float32 4MB 2.675e+...
    y_coordinate                                   (y, x) float32 4MB 2.492e+...
    is_spain                                       (y, x) uint16 2MB 0 0 ... 0 0
    AutonomousCommunities                          (y, x) uint16 2MB 0 0 ... 0 0
    ...                                             ...
    popdens_2015_mean                              (y, x) float32 4MB 0.0 ......
    popdens_2016_mean                              (y, x) float32 4MB 0.0 ......
    popdens_2017_mean                              (y, x) float32 4MB 0.0 ......
    popdens_2018_mean                              (y, x) float32 4MB 0.0 ......
    popdens_2019_mean                              (y, x) float32 4MB 0.0 ......
    popdens_2020_mean                              (y, x) float32 4MB 0.0 ......
Attributes:
    description:         Datacube for Spain with 1km x 1km spatial resolution...
    crs:                 EPSG:3035
    spatial_resolution:  1000.0 x 1000.0000000000002 (meters)
    top_left_corner:     2674734.3466, 2492195.9911

## 3. Add Temporal Variables:

### 3.1 IS FIRE (Class to predict)

In [ ]:
# Open fire instances dataset:
import pandas as pd
fire_instances = pd.read_csv("....../Processed data/EFFIS/Fire_instances_province_expanded_noGeometry.csv")

In [22]:
fire_instances

,Unnamed: 0,x_index,y_index,area_ha,cell_area,percentage_of_cell,date
0,0,577,850,104,29425.044828,0.029425,2008-06-17
1,1,578,850,104,153342.071315,0.153342,2008-06-17
2,2,577,851,104,448658.430933,0.448658,2008-06-17
3,3,578,851,104,301242.451535,0.301242,2008-06-17
4,4,577,852,104,99576.931626,0.099577,2008-06-17
...,...,...,...,...,...,...,...
76447,76445,534,149,18,29878.428259,0.029878,2024-12-30
76448,76430,534,150,18,151631.449336,0.151631,2024-12-30
76449,76455,301,463,4,44715.978185,0.044716,2024-12-30
76450,76446,534,149,18,29878.428259,0.029878,2024-12-31


In [23]:
# Add an empty variable for the "is_fire" variable

is_fire_data = np.zeros((len(ds.time), len(ds.y), len(ds.x)), dtype = np.uint8)

ds["is_fire"] = xr.DataArray(
    data=is_fire_data,
    coords={
        "time": ds.time,
        "y": ds.y,
        "x": ds.x
    },
    dims=("time", "y", "x"),
    name="is_fire"
)

ds["is_fire"].attrs = {
    "description": "Binary mask indicating the presence of fire in that cell (1) or no fire (0).",
    "valid_values": "{0, 1}",
    "data_type": "uint8",
}

In [24]:
import pandas as pd
fire_instances['date'] = pd.to_datetime(fire_instances['date'])
fire_instances = fire_instances.sort_values(by="date")
fire_instances

,Unnamed: 0,x_index,y_index,area_ha,cell_area,percentage_of_cell,date
0,0,577,850,104,29425.044828,0.029425,2008-06-17
1,1,578,850,104,153342.071315,0.153342,2008-06-17
2,2,577,851,104,448658.430933,0.448658,2008-06-17
3,3,578,851,104,301242.451535,0.301242,2008-06-17
4,4,577,852,104,99576.931626,0.099577,2008-06-17
...,...,...,...,...,...,...,...
76448,76430,534,150,18,151631.449336,0.151631,2024-12-30
76447,76445,534,149,18,29878.428259,0.029878,2024-12-30
76449,76455,301,463,4,44715.978185,0.044716,2024-12-30
76450,76446,534,149,18,29878.428259,0.029878,2024-12-31


In [25]:
fire_instances = fire_instances[fire_instances["area_ha"] > 5] # Only consider "big" fires

In [26]:
len(fire_instances)

74491

In [27]:
def days_since(input_date):
    """
    Given a date (as string, datetime, or similar),
    returns the number of days since 2008-01-01.
    """
    date = pd.to_datetime(input_date)
    ref_date = pd.to_datetime("2007-12-01")
    delta = date - ref_date
    return delta.days

calculate_fires = True

if calculate_fires:
    for i,(index, row) in enumerate(fire_instances.iterrows()):
        time_index = days_since(row["date"])
        x_index = row["x_index"]
        y_index = row["y_index"]
        ds["is_fire"][time_index][y_index][x_index] = 1

        if i % 1000 == 0:
            print(f"Iteration {i} / {len(fire_instances)}")


Iteration 0 / 74491
Iteration 1000 / 74491
Iteration 2000 / 74491
Iteration 3000 / 74491
Iteration 4000 / 74491
Iteration 5000 / 74491
Iteration 6000 / 74491
Iteration 7000 / 74491
Iteration 8000 / 74491
Iteration 9000 / 74491
Iteration 10000 / 74491
Iteration 11000 / 74491
Iteration 12000 / 74491
Iteration 13000 / 74491
Iteration 14000 / 74491
Iteration 15000 / 74491
Iteration 16000 / 74491
Iteration 17000 / 74491
Iteration 18000 / 74491
Iteration 19000 / 74491
Iteration 20000 / 74491
Iteration 21000 / 74491
Iteration 22000 / 74491
Iteration 23000 / 74491
Iteration 24000 / 74491
Iteration 25000 / 74491
Iteration 26000 / 74491
Iteration 27000 / 74491
Iteration 28000 / 74491
Iteration 29000 / 74491
Iteration 30000 / 74491
Iteration 31000 / 74491
Iteration 32000 / 74491
Iteration 33000 / 74491
Iteration 34000 / 74491
Iteration 35000 / 74491
Iteration 36000 / 74491
Iteration 37000 / 74491
Iteration 38000 / 74491
Iteration 39000 / 74491
Iteration 40000 / 74491
Iteration 41000 / 74491
Itera

#### Number of fire instances:

In [28]:
suma = ds["is_fire"].sum(dim="time")
suma.values.sum()

73811

In [29]:
ds

<xarray.Dataset> Size: 15GB
Dimensions:                                        (y: 920, x: 1188, time: 6241)
Coordinates:
  * x                                              (x) float64 10kB 2.675e+06...
  * y                                              (y) float64 7kB 2.492e+06 ...
  * time                                           (time) datetime64[ns] 50kB ...
Data variables: (12/234)
    x_index                                        (y, x) uint16 2MB 0 ... 1187
    y_index                                        (y, x) uint16 2MB 0 0 ... 919
    x_coordinate                                   (y, x) float32 4MB 2.675e+...
    y_coordinate                                   (y, x) float32 4MB 2.492e+...
    is_spain                                       (y, x) uint16 2MB 0 0 ... 0 0
    AutonomousCommunities                          (y, x) uint16 2MB 0 0 ... 0 0
    ...                                             ...
    popdens_2016_mean                              (y, x) float32 4MB 0.0 ......
    popdens_2017_mean                              (y, x) float32 4MB 0.0 ......
    popdens_2018_mean                              (y, x) float32 4MB 0.0 ......
    popdens_2019_mean                              (y, x) float32 4MB 0.0 ......
    popdens_2020_mean                              (y, x) float32 4MB 0.0 ......
    is_fire                                        (time, y, x) uint8 7GB 0 ....
Attributes:
    description:         Datacube for Spain with 1km x 1km spatial resolution...
    crs:                 EPSG:3035
    spatial_resolution:  1000.0 x 1000.0000000000002 (meters)
    top_left_corner:     2674734.3466, 2492195.9911

### 3.2 IS_NEAR_FIRE

In [ ]:

# Open fire instances dataset:
import pandas as pd
fire_instances = pd.read_csv("....../Processed data/EFFIS/Fire_instances_province_expanded_noGeometry.csv")

# Add an empty variable for the "is_fire" variable

is_near_fire_data = np.zeros((len(ds.time), len(ds.y), len(ds.x)), dtype = np.uint8)

ds["is_near_fire"] = xr.DataArray(
    data=is_near_fire_data,
    coords={
        "time": ds.time,
        "y": ds.y,
        "x": ds.x
    },
    dims=("time", "y", "x"),
    name="is_near_fire"
)

ds["is_near_fire"].attrs = {
    "description": "Binary mask indicating the presence of fire near that cell (1) or no (0).",
    "valid_values": "{0, 1}",
    "data_type": "uint8",
}

fire_instances['date'] = pd.to_datetime(fire_instances['date'])
fire_instances = fire_instances.sort_values(by="date")
fire_instances

spatial_range = 25
temporal_range = 10


for m, (_, row) in enumerate(fire_instances.iterrows()):
    time_index = days_since(row["date"])
    x_index = row["x_index"]
    y_index = row["y_index"]

    t_start = time_index - temporal_range
    t_end = time_index + 1

    y_start = max(0, y_index - spatial_range + 1)
    y_end = min(y_index + spatial_range, ds.sizes['y'])
    x_start = max(0, x_index - spatial_range + 1)
    x_end = min(x_index + spatial_range, ds.sizes['x'])
    
    ds["is_near_fire"].values[t_start:t_end, y_start:y_end, x_start:x_end] = 1

    if m % 1000 == 0:
        print(f"Iteration {m} / {len(fire_instances)}")

Iteration 0 / 76452
Iteration 1000 / 76452
Iteration 2000 / 76452
Iteration 3000 / 76452
Iteration 4000 / 76452
Iteration 5000 / 76452
Iteration 6000 / 76452
Iteration 7000 / 76452
Iteration 8000 / 76452
Iteration 9000 / 76452
Iteration 10000 / 76452
Iteration 11000 / 76452
Iteration 12000 / 76452
Iteration 13000 / 76452
Iteration 14000 / 76452
Iteration 15000 / 76452
Iteration 16000 / 76452
Iteration 17000 / 76452
Iteration 18000 / 76452
Iteration 19000 / 76452
Iteration 20000 / 76452
Iteration 21000 / 76452
Iteration 22000 / 76452
Iteration 23000 / 76452
Iteration 24000 / 76452
Iteration 25000 / 76452
Iteration 26000 / 76452
Iteration 27000 / 76452
Iteration 28000 / 76452
Iteration 29000 / 76452
Iteration 30000 / 76452
Iteration 31000 / 76452
Iteration 32000 / 76452
Iteration 33000 / 76452
Iteration 34000 / 76452
Iteration 35000 / 76452
Iteration 36000 / 76452
Iteration 37000 / 76452
Iteration 38000 / 76452
Iteration 39000 / 76452
Iteration 40000 / 76452
Iteration 41000 / 76452
Itera

In [ ]:
# Create an encoding dictionary for all data variables
encoding = {var: {"zlib": True, "complevel": 5} for var in ds.data_vars}
ds.to_netcdf("......../Datacube/Base_datacube.nc",
              encoding=encoding)